# TradingAgents 实验回测

这里回测五项实验：比例仓位、新闻情绪、做空、多轮辩论和单 Agent。每项实验从 CSV 读取目标仓位，按信号覆盖的日期回测并单独保存结果。

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

from agents_single import validate_single_agent
from data import read_prices
from signals import load_decisions, validate_weights
from settings import load_config
from backtest import (
    build_baselines, save_baselines, load_baselines, load_original, original_frames,
    select_period, prepare_backtest, compare_positions, plot_backtest,
    monthly_strategy_returns, save_results,
)

ROOT = Path.cwd()


## 参数与实际信号日期

VARIANT_CSVS 选择要跑的实验。路径相对于 data/decisions，首层目录名是实验名。单 Agent 的信号文件到位后，自动加入本次实验回测。EXCLUDE_PERIODS=[] 使用完整区间。

In [ ]:
cfg = load_config()
TICKERS = cfg["tickers"]
AGENT_CSV = ROOT / "data/decisions/ta-v0.6.0-deepseek-flash-market-news-fundamentals-main-2026jul-oct-v1.csv"
AGENT_NAME = "TradingAgents (3 analysts)"
VARIANT_CSVS = {
    "position/position_aug.csv": "Position sizing",
    "sentiment/sentiment_aug.csv": "News sentiment",
    "short/short_approval_aug.csv": "Short approval",
    "short/short_mapping.csv": "Short rating mapping",
    "debate/debate_r1_sep.csv": "Debate (1 round)",
    "debate/debate_r3_sep.csv": "Debate (3 rounds)",
}  # data/decisions/ 下的路径，首层目录名即实验名
SINGLE_AGENT_CSV = "single/single_agent.csv"
if (ROOT / "data/decisions" / SINGLE_AGENT_CSV).is_file():
    VARIANT_CSVS[SINGLE_AGENT_CSV] = "Single Agent"
if not VARIANT_CSVS:
    raise ValueError("请在 VARIANT_CSVS 中至少选择一个变体")
RETURN_TYPE = "simple"
NAV_MODE = "additive"  # 保留原净值口径；可改为 compound

# 特殊日期筛选：起止日均包含，可追加多段；设为 [] 恢复完整回测。
# 与区间相交的收盘到下一收盘收益会被排除，所有股票和策略使用同一筛选。
EXCLUDE_PERIODS = [("2026-07-30", "2026-08-03")] #("2026-07-30", "2026-08-03")

# 首个信号日收盘开始评分；完整覆盖由后面的校验检查。
signal_input = pd.read_csv(AGENT_CSV, keep_default_na=False)
signal_dates = sorted(signal_input.information_cutoff.str[:10].unique())
seed_date = signal_dates[0]
backtest_start, backtest_end = signal_dates[0], signal_dates[-1]
periods = {"agent_period": (backtest_start, backtest_end)}
period_label = "Agent signal period"
if EXCLUDE_PERIODS:
    period_label += "\nExcluded: " + ", ".join(f"{a} to {b}" for a, b in EXCLUDE_PERIODS)
period_names = {"agent_period": period_label}
pd.DataFrame(periods, index=["start", "end"]).T


## 1. 读取行情
保留完整历史用于指标预热；收盘到下一收盘收益在合并信号时统一构造。


In [ ]:
# 完整历史用于SMA/MACD预热，收益只在实际信号期间计算。
price_histories = {ticker: read_prices(ticker, cfg, backtest_end) for ticker in TICKERS}
pd.DataFrame([
    {"ticker": ticker, "first_date": prices.date.iloc[0],
     "last_date": prices.date.iloc[-1], "rows": len(prices)}
    for ticker, prices in price_histories.items()
])


## 2. 生成同期规则信号并保存CSV
Buy and Hold、SMA、MACD与Agent共用日期和种子日。先落盘，再读取回测。


In [ ]:
baseline_tables = build_baselines(price_histories, cfg, backtest_start, backtest_end)


In [ ]:
baseline_paths = save_baselines(baseline_tables, ROOT, backtest_start, backtest_end, cfg)
pd.Series(baseline_paths, name="signals_csv")


## 3. 读取信号 CSV

原版和规则基线读取 BUY／SELL／HOLD 动作。各实验读取 ticker、signal_date、target_weight。单 Agent 的权重只接受 0 或 1。

In [ ]:
decision_tables = load_baselines(baseline_paths, backtest_start, backtest_end)


In [ ]:
strategy_names = {
    "buy_hold": "Buy and Hold",
    "sma": f"SMA Trend ({cfg['sma']['fast']}/{cfg['sma']['slow']})",
    "macd": f"MACD Trend ({cfg['macd']['fast']}/{cfg['macd']['slow']}/{cfg['macd']['signal']})",
}

decision_tables.update(load_original(AGENT_CSV, TICKERS, backtest_start, backtest_end))
strategy_names["agent"] = AGENT_NAME
variant_tables = {}
variant_paths = {}
for relative_path, label in VARIANT_CSVS.items():
    signal_path = Path(relative_path)
    if (signal_path.is_absolute() or len(signal_path.parts) < 2
            or ".." in signal_path.parts or signal_path.parts[0] == "original"):
        raise ValueError(f"请使用 data/decisions/ 下的实验子目录路径：{relative_path}")
    path = ROOT / "data/decisions" / signal_path
    strategy = path.stem
    if strategy in strategy_names:
        raise ValueError(f"重复策略名：{strategy}")
    strategy_names[strategy] = label
    variant_paths[strategy] = signal_path
    table = load_decisions(path)
    if table.empty or "ticker" not in table or not set(table.ticker).issubset(TICKERS):
        raise ValueError(f"{strategy} 缺少信号或含未知股票")
    for ticker, rows in table.groupby("ticker"):
        if signal_path.parts[0] == "single":
            variant_tables[ticker, strategy] = validate_single_agent(rows, ticker)
        else:
            variant_tables[ticker, strategy] = validate_weights(rows, ticker)
all_tables = decision_tables | variant_tables
pd.DataFrame([{"ticker": ticker, "strategy": strategy, "strategy_name": strategy_names[strategy],
               "decision_rows": len(rows)} for (ticker, strategy), rows in all_tables.items()])


## 4. 对齐信号日收盘与收益区间

目标权重在这里截断到 [-1, 1]。短样本按已有信号覆盖的成交日至最终估值日单独比较，同期原版和规则基线重新从净值 1、仓位 0 开始计费。

In [ ]:
base_frames = original_frames(price_histories, decision_tables, backtest_start, backtest_end, RETURN_TYPE)
position_tables = {}
comparison_periods = dict(periods)
for (ticker, strategy), rows in variant_tables.items():
    prices = price_histories[ticker]
    days = prices.date.tolist()
    first = days.index(rows.signal_date.iloc[0])
    last = min(days.index(rows.signal_date.iloc[-1]) + 1, len(days) - 1)
    start, end = days[first], days[last]
    if not backtest_start <= start < end <= backtest_end:
        raise ValueError(f"{ticker} / {strategy} 不在原版可比较区间内")
    period = ("agent_period" if (start, end) == (backtest_start, backtest_end)
              else f"{start}_to_{end}")
    comparison_periods[period] = (start, end)
    period_names.setdefault(period, "Matched signal period")
    matched_prices = prices.iloc[first:last + 1]
    position_tables[ticker, period, strategy] = prepare_backtest(
        matched_prices, rows, return_type=RETURN_TYPE)
    for name in ("agent", "buy_hold", "sma", "macd"):
        key = (ticker, period, name)
        if key not in position_tables:
            position_tables[key] = select_period(base_frames[ticker, name], start, end)

pd.concat({key: frame[["close", "signal", "position", "pct"]].head(3)
           for key, frame in position_tables.items()}, names=["ticker", "period", "strategy"])

## 统一回测口径

两本 Notebook 从 backtest.py 导入同一套函数。仓位按信号日收盘对齐，期收益为仓位乘资产收益，扣除相邻仓位差绝对值乘费用。加法净值从 1 开始累计；筛选在计算完整收益和换手之后执行。

## 同期策略比较
按股票比较Agent与三条规则基线的收益、Sharpe、回撤与换手。


In [ ]:
fee = (cfg["commission_bps"] + cfg["slippage_bps"]) / 10000
backtests, metrics_table = compare_positions(
    position_tables, strategy_names, cfg, fee, RETURN_TYPE, NAV_MODE, EXCLUDE_PERIODS)
metrics_table


## 按实验绘制净值与回撤

每个实验单独显示，与原版和规则基线使用相同股票、收益日期和费用。不同评价区间分别画图。

In [ ]:
# 同一批回测共用收益计算，只按实验归档、绘图。
baseline_names = ("agent", "buy_hold", "sma", "macd")
experiment_cases = {}
for (ticker, period, strategy), frame in backtests.items():
    if strategy in variant_paths:
        experiment = variant_paths[strategy].parts[0]
        cases = experiment_cases.setdefault(experiment, {})
        frames = cases.setdefault((ticker, period), {
            name: backtests[ticker, period, name] for name in baseline_names
        })
        frames[strategy] = frame

figures = {}
for experiment, cases in experiment_cases.items():
    for (ticker, period), frames in cases.items():
        metrics = metrics_table.xs((ticker, period), level=("ticker", "period"))
        metrics = metrics.loc[list(frames)]
        fig, curves = plot_backtest(frames, strategy_names, metrics,
                                   f"{ticker} / {experiment} / {period_names[period]}",
                                   nav_mode=NAV_MODE)
        figures[experiment, ticker, period] = fig
        plt.show()
        plt.close(fig)


## 月收益与按实验导出

回测输出保存在 `results/<实验>/full/` 或 `results/<实验>/filtered/`。每组包括指标、月收益、逐期收益与换手、净值和回撤图以及本次运行参数。

In [ ]:
monthly_returns = monthly_strategy_returns(backtests, NAV_MODE)
monthly_returns.to_frame()


In [ ]:
mode = "filtered" if EXCLUDE_PERIODS else "full"
output_dirs = {}

for experiment, cases in experiment_cases.items():
    output_dir = ROOT / "results" / experiment / mode
    strategies = {strategy: strategy_names[strategy]
                  for frames in cases.values() for strategy in frames}
    record = {
        "experiment": experiment, "tickers": sorted({ticker for ticker, _ in cases}),
        "periods": {period: comparison_periods[period] for _, period in cases},
        "strategies": strategies,
        "agent_csv": str(AGENT_CSV.relative_to(ROOT)),
        "baseline_csvs": {ticker: str(baseline_paths[ticker].relative_to(ROOT))
                          for ticker, _ in cases},
        "variant_csvs": [str(path) for path in variant_paths.values() if path.parts[0] == experiment],
        "price_basis": "signal_close_to_next_close", "return_type": RETURN_TYPE, "nav_mode": NAV_MODE, "fee": fee, "exclude_periods": EXCLUDE_PERIODS,
    }
    case_figures = {(ticker, period): figures[experiment, ticker, period] for ticker, period in cases}
    save_results(output_dir, cases, metrics_table, monthly_returns, case_figures, record)
    output_dirs[experiment] = output_dir

pd.Series(output_dirs, name="回测输出目录")
